# CIFAR-100 — ResNet-50 Two-Phase Fine-Tuning

Implements **two-phase fine-tuning** on a pre-trained ResNet-50 to squeeze more accuracy
from transfer learning on CIFAR-100.

## Why Two-Phase?

Standard fine-tuning trains all layers simultaneously. The problem: the FC head is
**randomly initialised** and produces large, noisy gradients that flow back into the
pretrained backbone and corrupt its features in early epochs.

**Two-phase fixes this:**

| Phase | Backbone | Head | LR | Epochs | Goal |
|-------|----------|------|----|--------|------|
| 1 — Warm-up | **Frozen** | Trainable | 0.05 | 10 | Stable head without corrupting backbone |
| 2 — Fine-tune | **Unfrozen** | Trainable | 0.005 | 50 | Full network fine-tuning at safe LR |


## Imports

In [1]:
import sys

from ipykernel.kernelapp import kernel_flags

sys.path.append('../..')

import torch
import torch.nn as nn
import torchvision.models as models
from torch import optim
from torch.amp import GradScaler

from utils.dataset import get_cifar100_dataloaders, CIFAR100_CLASSES
from utils.training import fit, test_accuracy
from utils.callbacks import ModelCheckpoint
from utils.plotting import plot_training_curves, show_sample_batch

## Device Setup

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
print(f'Using device: {device}')

Using device: cuda


## Data

Same as standard ResNet-50 transfer: 224×224 with ImageNet stats.


In [3]:
batch_size = 64

trainloader, valloader, testloader = get_cifar100_dataloaders(
    batch_size=batch_size,
    num_workers=4,
    img_size=224,
    use_imagenet_stats=True,
)
print(f'Train: {len(trainloader)} | Val: {len(valloader)} | Test: {len(testloader)}')

C:\Users\asmit\PycharmProjects\CIFAR_10\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 625 | Val: 157 | Test: 157


## Model

In [4]:
class SEBlock(nn.Module):
    def __init__(self, c, r=16):
        super(SEBlock, self).__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(c, c//r, bias=False),
            nn.ReLU(),
            nn.Linear(c//r, c, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.se(x).view(x.size(0), x.size(1), 1, 1)

In [5]:
class DepthwiseSeperableConv(nn.Module):
    """V1 baseline: no BN/SiLU after depthwise conv."""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1):
        super(DepthwiseSeperableConv, self).__init__()
        self.depthwise = nn.Sequential(
            nn.Conv2d(
                in_channels,
                in_channels,
                kernel_size,
                stride,
                padding=kernel_size // 2,
                groups=in_channels,
                bias=False
            ),
        )

        self.se = SEBlock(in_channels)

        self.pointwise = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        x = self.depthwise(x)
        x = self.se(x)
        x = self.pointwise(x)
        return x

In [6]:
class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, kernel_size=3, t=6):
        super(MBConv, self).__init__()
        self.expansion = nn.Sequential(
            nn.Conv2d(in_channels, in_channels * t, kernel_size=1),
            nn.BatchNorm2d(in_channels * t),
            nn.SiLU()
        )
        self.depthwise = nn.Sequential(
            DepthwiseSeperableConv(in_channels * t, out_channels, kernel_size, stride),
        )

        self.res = nn.Identity()
        self.use_residual = (
            stride == 1 and in_channels == out_channels
        )

    def forward(self, x):
        identity = x
        x = self.expansion(x)
        x = self.depthwise(x)
        if self.use_residual:
            x += identity
        return x

In [7]:
class EfficientNet(nn.Module):
    def __init__(self, num_classes=100, t=6):
        super(EfficientNet, self).__init__()
        self.t = t
        self.conv1 = self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.silu = nn.SiLU()
        self.bn1 = self.bn1 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2)
        self.layer1 = self._make_layer(1, 32, 16)
        self.layer2 = self._make_layer(2, 16, 24)
        self.layer3 = self._make_layer(2, 24, 40, kernel_size=5)
        self.layer4 = self._make_layer(3, 40, 80)
        self.layer5 = self._make_layer(3, 80, 112, kernel_size=5)
        self.layer6 = self._make_layer(4, 112, 192, kernel_size=5, stride=1)
        self.layer7 = self._make_layer(1, 192, 320)
        self.conv2 = nn.Conv2d(320, 1280, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn2 = nn.BatchNorm2d(1280)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(1280, num_classes)

    def _make_layer(self, blocks, in_channels, out_channels, kernel_size=3, stride=2):
        layers = [MBConv(in_channels, out_channels, stride=stride, kernel_size=kernel_size, t=self.t)]

        for _ in range(1, blocks):
            layers.append(MBConv(out_channels, out_channels, kernel_size=kernel_size, t=self.t))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.silu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = self.layer6(x)
        x = self.layer7(x)
        x = self.bn2(self.conv2(x))
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [8]:
model = EfficientNet().to(device)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

EfficientNet(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (silu): SiLU()
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): MBConv(
      (expansion): Sequential(
        (0): Conv2d(32, 192, kernel_size=(1, 1), stride=(1, 1))
        (1): BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU()
      )
      (depthwise): Sequential(
        (0): DepthwiseSeperableConv(
          (depthwise): Sequential(
            (0): Conv2d(192, 192, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=192, bias=False)
          )
          (se): SEBlock(
            (se): Sequential(
              (0): AdaptiveAvgPool2d(output_size=1)
              (1): Flatten(start_dim=1, end_dim=-1)
              (2): Linear(in_features=192, out_features=12, bias=F

In [9]:
EPOCHS = 120

criterion = nn.CrossEntropyLoss()
scaler    = GradScaler('cuda')

optimizer = optim.SGD(
    model.parameters(),
    lr=0.05,
    momentum=0.9,
    weight_decay=1e-4,
    nesterov=True,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-4,
)
MODEL_NAME = 'efficient_net_100_v1'
checkpoint = ModelCheckpoint(
    model,
    checkpoint_path=f'../../checkpoint/{MODEL_NAME}_last.pth',
    best_model_path=f'../../checkpoint/{MODEL_NAME}_best.pth',
    mode='max',
    verbose=True,
)

In [10]:
# ── Resume from checkpoint ───────────────────────────────────────────────
ckpt = torch.load('../../checkpoint/efficient_net_100_last.pth', map_location=device, weights_only=False)

start_epoch = ckpt['epoch'] + 1          
model.load_state_dict(ckpt['model_state'])
optimizer.load_state_dict(ckpt['optimizer_state'])
scheduler.load_state_dict(ckpt['scheduler_state'])
scaler.load_state_dict(ckpt['scaler_state'])

print(f"Resumed — continuing from epoch {start_epoch + 1}/{EPOCHS}")


RuntimeError: Error(s) in loading state_dict for EfficientNet:
	Missing key(s) in state_dict: "layer1.0.depthwise.0.depthwise.0.weight", "layer1.0.depthwise.0.pointwise.0.weight", "layer1.0.depthwise.0.pointwise.1.weight", "layer1.0.depthwise.0.pointwise.1.bias", "layer1.0.depthwise.0.pointwise.1.running_mean", "layer1.0.depthwise.0.pointwise.1.running_var", "layer2.0.depthwise.0.depthwise.0.weight", "layer2.0.depthwise.0.pointwise.0.weight", "layer2.0.depthwise.0.pointwise.1.weight", "layer2.0.depthwise.0.pointwise.1.bias", "layer2.0.depthwise.0.pointwise.1.running_mean", "layer2.0.depthwise.0.pointwise.1.running_var", "layer2.1.depthwise.0.depthwise.0.weight", "layer2.1.depthwise.0.pointwise.0.weight", "layer2.1.depthwise.0.pointwise.1.weight", "layer2.1.depthwise.0.pointwise.1.bias", "layer2.1.depthwise.0.pointwise.1.running_mean", "layer2.1.depthwise.0.pointwise.1.running_var", "layer3.0.depthwise.0.depthwise.0.weight", "layer3.0.depthwise.0.pointwise.0.weight", "layer3.0.depthwise

In [ ]:
train_losses, val_losses, val_accs = fit(
    model, trainloader, valloader, criterion,
    optimizer, scheduler, scaler, device,
    EPOCHS, checkpoint,
    step_scheduler_per_batch=False,
)

## Combined Training Curves

In [ ]:
# Combine both phases for a single vie

plot_training_curves(train_losses, val_losses, val_accs)

## Test Evaluation

Load the best checkpoint across **both phases** and evaluate.


In [ ]:
checkpoint.restore_best_weights()

overall_acc, per_class = test_accuracy(model, testloader, CIFAR100_CLASSES, device)
print(f'Test Accuracy: {overall_acc:.2f}%\n')

print('Per-class accuracies (best first):')
sorted_classes = sorted(per_class.items(), key=lambda x: x[1], reverse=True)
for cls, acc in sorted_classes:
    bar = '█' * int(acc / 5)
    print(f'  {cls:<20s} {acc:5.1f}%  {bar}')